In [21]:
import ee
import geemap
import geopandas as gpd 
import pandas as pd
import numpy as np
import rasterio
from shapely import wkt 
from tqdm.auto import tqdm
from shapely.geometry.base import BaseGeometry
from shapely.geometry import box, mapping
from pathlib import Path
from rasterio.merge import merge 
from rasterio.enums import Resampling
from rasterio.mask import mask
from rasterio.mask import mask
from collections import defaultdict


In [22]:
# 1) Authenticate / initialize (run Authenticate only the first time)
#try:
    #ee.Initialize()
#except Exception:
    #ee.Authenticate()
    #ee.Initialize()


In [68]:
#Loading the files

# GEDI footprint pairs
pairs = gpd.read_file("GEDI_Pairs/GEDI_Footprint_Pairs_Fire_with_FF.gpkg")
pairs["pair_uid"] = pairs.index.astype(int)

# Parse WKT -> shapely geometry (for pre/post columns)
pairs["pre_geom"] = pairs["pre_geom"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)
pairs["post_geom"] = pairs["post_geom"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)



In [69]:
# 2) Create 10 m buffers using pairing_crs (meters), then back to EPSG:4326
buffer_size = 10
pairs["pre_buff_10"] = None
pairs["post_buff_10"] = None

for pcrs, idxs in tqdm(pairs.groupby("pairing_crs").groups.items(), desc="Buffering by pairing_crs"):
    if pd.isna(pcrs):
        continue

    pre_ll = gpd.GeoSeries(pairs.loc[idxs, "pre_geom"], crs="EPSG:4326")
    post_ll = gpd.GeoSeries(pairs.loc[idxs, "post_geom"], crs="EPSG:4326")

    pairs.loc[idxs, "pre_buff_10"] = pre_ll.to_crs(pcrs).buffer(buffer_size).to_crs("EPSG:4326").values
    pairs.loc[idxs, "post_buff_10"] = post_ll.to_crs(pcrs).buffer(buffer_size).to_crs("EPSG:4326").values



Buffering by pairing_crs:   0%|          | 0/3 [00:00<?, ?it/s]

In [70]:
# Build one footprints table (pre + post)
pre_gdf = gpd.GeoDataFrame(pairs[["pair_uid"]].copy(), geometry=pairs["pre_buff_10"], crs="EPSG:4326")
pre_gdf["footprint"] = "pre"

post_gdf = gpd.GeoDataFrame(pairs[["pair_uid"]].copy(), geometry=pairs["post_buff_10"], crs="EPSG:4326")
post_gdf["footprint"] = "post"

footprints = pd.concat([pre_gdf, post_gdf], ignore_index=True)
footprints = gpd.GeoDataFrame(footprints, geometry="geometry", crs="EPSG:4326")
footprints = footprints[footprints.geometry.notna() & ~footprints.geometry.is_empty].copy()

print("footprints rows:", len(footprints))
print("footprints unique pair_uid:", footprints["pair_uid"].nunique())


footprints rows: 52480
footprints unique pair_uid: 26240


In [ ]:
# Loop through tiles and accumulate all fire years per (pair_uid, footprint)
tiles_dir = Path(r"Fire History/Fire Years")
tile_paths = sorted(tiles_dir.glob("*.tif"))
print("tiles:", len(tile_paths))

years_acc = defaultdict(set)
fp_cache = {} 

for tile in tqdm(tile_paths, desc="Processing tiles"):
    with rasterio.open(tile) as ds:
        crs_key = str(ds.crs)

        if crs_key not in fp_cache:
            fp_cache[crs_key] = footprints.to_crs(ds.crs) if footprints.crs != ds.crs else footprints.copy()
            fp_cache[crs_key].sindex  # build spatial index

        fp = fp_cache[crs_key]
        tile_poly = box(*ds.bounds)

        idx = list(fp.sindex.query(tile_poly, predicate="intersects"))
        if not idx:
            continue

        subset = fp.iloc[idx]

        for r in subset.itertuples(index=False):
            try:
                out, _ = mask(ds, [mapping(r.geometry)], crop=True, filled=False, all_touched=True)
            except Exception:
                continue

            vals = out.compressed() if np.ma.isMaskedArray(out) else out.ravel()
            if ds.nodata is not None:
                vals = vals[vals != ds.nodata]
            vals = vals[~np.isnan(vals)]
            vals = vals[vals > 0]

            if vals.size == 0:
                continue

            years_acc[(int(r.pair_uid), r.footprint)].update(vals.astype(int).tolist())


tiles: 56


Processing tiles:   0%|          | 0/56 [00:00<?, ?it/s]

In [ ]:
# Convert accumulator -> DataFrame
# Accumulator has the information of all the fire_years

records = []
for (pid, fp), ys in years_acc.items():
    ys = sorted(ys)
    records.append({
        "pair_uid": pid,
        "footprint": fp,
        "fire_years": ys,
        "fire_years_str": ",".join(map(str, ys))
    })

fire_years_long = pd.DataFrame(records)
print("rows:", len(fire_years_long))
print("unique pair_uid:", fire_years_long["pair_uid"].nunique() if len(fire_years_long) else 0)


rows: 48268
unique pair_uid: 24169


In [73]:
# Map back to pairs
pre_map = fire_years_long.loc[fire_years_long["footprint"] == "pre"].set_index("pair_uid")["fire_years_str"].to_dict()
post_map = fire_years_long.loc[fire_years_long["footprint"] == "post"].set_index("pair_uid")["fire_years_str"].to_dict()

pairs["pre_fire_years"] = pairs["pair_uid"].map(pre_map).fillna("")
pairs["post_fire_years"] = pairs["pair_uid"].map(post_map).fillna("")

print("non-empty pre:", (pairs["pre_fire_years"] != "").sum())
print("non-empty post:", (pairs["post_fire_years"] != "").sum())
pairs[["pair_uid", "pre_fire_years", "post_fire_years"]].head()


non-empty pre: 24132
non-empty post: 24136


,pair_uid,pre_fire_years,post_fire_years
0,0,2020,2020
1,1,2021,2021
2,2,"2020,2022","2020,2022"
3,3,"2022,2023","2022,2023"
4,4,2022,2022


In [ ]:
#Check the numbers and overall results

print("total pairs:", len(pairs))
print("pairs with any fire (pre or post):", ((pairs["pre_fire_years"] != "") | (pairs["post_fire_years"] != "")).sum())
print("pairs with both sides fire:", ((pairs["pre_fire_years"] != "") & (pairs["post_fire_years"] != "")).sum())

# sample rows that do have values
pairs.loc[(pairs["pre_fire_years"] != "") | (pairs["post_fire_years"] != ""),
          ["pair_uid", "pre_fire_years", "post_fire_years"]].head(10)


total pairs: 26240
pairs with any fire (pre or post): 24169
pairs with both sides fire: 24099


,pair_uid,pre_fire_years,post_fire_years
0,0,2020,2020
1,1,2021,2021
2,2,"2020,2022","2020,2022"
3,3,"2022,2023","2022,2023"
4,4,2022,2022
5,5,"1999,2014,2018,2022","1999,2014,2018,2022"
6,6,"2020,2022","2020,2022"
7,7,"2020,2022","2020,2022"
8,8,2021,2021
9,9,2022,2022


In [ ]:
# Deletes buffer columns and transforms pre_geom and post_geom back to only text (not geometries)

pairs_out = pairs.drop(columns=["pre_buff_10", "post_buff_10"], errors="ignore").copy()
pairs_out["pre_geom"] = gpd.GeoSeries(pairs_out["pre_geom"]).to_wkt()
pairs_out["post_geom"] = gpd.GeoSeries(pairs_out["post_geom"]).to_wkt()

In [76]:
#Adjusting the dataset:
#  - Storing just fire years before the "fire_date" from the pairs
#  - Keeping only the pairs in which pre and post fire frequency and fire years are the same

# Parse fire_date and get cutoff year
pairs["fire_date_dt"] = pd.to_datetime(pairs["fire_date"], errors="coerce")
pairs["fire_date_year"] = pairs["fire_date_dt"].dt.year

# Keep only fire years BEFORE fire_date year
def years_before_cutoff(years_value, cutoff_year):
    if pd.isna(cutoff_year):
        return ""
    cutoff_year = int(cutoff_year)

    if years_value is None or (isinstance(years_value, float) and np.isnan(years_value)):
        return ""

    if isinstance(years_value, list):
        years = [int(y) for y in years_value if pd.notna(y)]
    else:
        s = str(years_value).strip()
        if s == "":
            return ""
        years = []
        for token in s.split(","):
            token = token.strip()
            if token.isdigit():
                years.append(int(token))

    kept = sorted(set(y for y in years if y < cutoff_year))
    return ",".join(map(str, kept))

pairs["pre_fire_years_before"] = [
    years_before_cutoff(y, c) for y, c in zip(pairs["pre_fire_years"], pairs["fire_date_year"])
]
pairs["post_fire_years_before"] = [
    years_before_cutoff(y, c) for y, c in zip(pairs["post_fire_years"], pairs["fire_date_year"])
]

# Keep only rows where pre/post are equal for FF and filtered fire years
mask_same = (
    pairs["pre_FF"].fillna(-9999).eq(pairs["post_FF"].fillna(-9999)) &
    pairs["pre_fire_years_before"].fillna("").eq(pairs["post_fire_years_before"].fillna(""))
)

pairs_final = pairs.loc[mask_same].copy()

# Collapse pre/post into single columns
pairs_final["fire_frequency"] = pairs_final["pre_FF"]
pairs_final["fire_years"] = pairs_final["pre_fire_years"] 
pairs_final["fire_years_before_fire_date"] = pairs_final["pre_fire_years_before"]

# Drop no-longer-needed columns
pairs_final = pairs_final.drop(columns=[
    "pre_FF", "post_FF",
    "pre_fire_years", "post_fire_years", 
    "post_fire_years_before", "pre_fire_years_before"
], errors="ignore")

print("Rows kept:", len(pairs_final))
pairs_final[["pair_uid", "fire_date", "fire_frequency", "fire_years","fire_years_before_fire_date"]].head()


Rows kept: 22280


,pair_uid,fire_date,fire_frequency,fire_years,fire_years_before_fire_date
0,0,2020-04-01,1,2020,
1,1,2021-11-01,1,2021,
2,2,2020-07-01,2,"2020,2022",
3,3,2023-11-01,2,"2022,2023",2022
4,4,2022-09-01,1,2022,


In [ ]:
#Getting the last fire year before the collection of Footprint 1 (pre-fire)
#Calculates the vegetation/biomass age after fire

#Convert into datetime
pairs_final["time_1_dt"] = pd.to_datetime(pairs_final["time_1"], errors="coerce")

def last_fire_before_time1(fire_years_str, time1_dt, fire_date_dt):
    if pd.isna(time1_dt):
        return np.nan

    s = "" if pd.isna(fire_years_str) else str(fire_years_str).strip()
    if s == "":
        return np.nan

    years = [int(t.strip()) for t in s.split(",") if t.strip().isdigit()]
    if not years:
        return np.nan

    valid = [y for y in years if y < time1_dt.year]  
    if pd.notna(fire_date_dt):
        valid = [y for y in valid if y < fire_date_dt.year]

    return max(valid) if valid else np.nan

pairs_final["last_fire_year_for_time1"] = [
    last_fire_before_time1(fy, t1, fd)
    for fy, t1, fd in zip(pairs_final["fire_years"], pairs_final["time_1_dt"], pairs_final["fire_date_dt"])
]

#Calculating the time after te last year of fire (age of vegetation in Footrprint 1)
pairs_final["pre_veg_age"] = pairs_final["time_1_dt"].dt.year - pairs_final["last_fire_year_for_time1"]
pairs_final.loc[pairs_final["pre_veg_age"] < 0, "pre_veg_age"] = np.nan


pairs_final[["pair_uid", "time_1", "fire_years", "last_fire_year_for_time1", "pre_veg_age"]].head()

,pair_uid,time_1,fire_years,last_fire_year_for_time1,pre_veg_age
0,0,2020-01-26 05:35:53.896,2020,NaN,NaN
1,1,2019-09-30 04:18:42.182,2021,NaN,NaN
2,2,2020-04-21 19:26:06.158,"2020,2022",NaN,NaN
3,3,2020-08-01 03:18:01.761,"2022,2023",NaN,NaN
4,4,2020-09-21 07:02:58.924,2022,NaN,NaN


In [47]:
print(pairs_final.columns)

Index(['index_1', 'index_2', 'distance_m', 'pre_index', 'post_index', 'time_1',
       'time_2', 'fire_date', 'post_fire_months', 'pre_agbd', 'post_agbd',
       'delta_agbd', 'pre_geom', 'post_geom', 'source_file', 'pairing_crs',
       'geometry', 'pair_uid', 'pre_buff_10', 'post_buff_10', 'fire_date_dt',
       'fire_date_year', 'fire_frequency', 'fire_years',
       'fire_years_before_fire_date', 'time_1_dt', 'last_fire_year_for_time1',
       'pre_veg_age'],
      dtype='object')


In [ ]:
## Adding the year of fire_date to fire_years, in case they do not exist there. 
# This is specially important for 2024 years, that are not included in the raster data (needs to be updated). 

def parse_years(s):
    if pd.isna(s):
        return []
    s = str(s).strip()
    if s == "":
        return []
    return sorted(set(int(t.strip()) for t in s.split(",") if t.strip().isdigit()))

# 1) Check how many pairs already contain fire_date year in fire_years
years_lists = pairs_final["fire_years"].apply(parse_years)

has_fire_date_year = [
    (not pd.isna(fd)) and (int(fd) in ys)
    for fd, ys in zip(pairs_final["fire_date_year"], years_lists)
]

print("Pairs where fire_date year is already in fire_years:", int(np.sum(has_fire_date_year)))
print("Pairs where it is missing:", int(len(pairs_final) - np.sum(has_fire_date_year)))

# 2) If missing, add fire_date year to fire_years
def ensure_fire_date_year_in_list(fire_years_str, fire_date_year):
    ys = parse_years(fire_years_str)
    if pd.notna(fire_date_year):
        ys = sorted(set(ys + [int(fire_date_year)]))
    return ",".join(map(str, ys))

missing_mask = ~pd.Series(has_fire_date_year, index=pairs_final.index)

pairs_final.loc[missing_mask, "fire_years"] = [
    ensure_fire_date_year_in_list(fy, fd)
    for fy, fd in zip(
        pairs_final.loc[missing_mask, "fire_years"],
        pairs_final.loc[missing_mask, "fire_date_year"]
    )
]

print("Updated rows:", int(missing_mask.sum()))


Pairs where fire_date year is already in fire_years: 19096
Pairs where it is missing: 3184
Updated rows: 3184


In [79]:
pairs.head()

,index_1,index_2,distance_m,pre_index,post_index,time_1,time_2,fire_date,post_fire_months,pre_agbd,...,geometry,pair_uid,pre_buff_10,post_buff_10,pre_fire_years,post_fire_years,fire_date_dt,fire_date_year,pre_fire_years_before,post_fire_years_before
0,36287,36288,2.041762,36287,36288,2020-01-26 05:35:53.896,2020-07-02 02:39:59.786,2020-04-01,3.0,147.778152,...,"LINESTRING (-49.64374 -4.61424, -49.64372 -4.6...",0,POLYGON ((-49.64348686753057 -4.61425499999397...,POLYGON ((-49.6434708675323 -4.614245999993978...,2020,2020,2020-04-01,2020,,
1,31053,31074,2.358854,31053,31074,2019-09-30 04:18:42.182,2022-08-07 23:45:11.162,2021-11-01,9.0,108.444229,...,"LINESTRING (-50.25587 -3.12689, -50.25586 -3.1...",1,POLYGON ((-50.2556191841294 -3.126911672252552...,POLYGON ((-50.255614184131296 -3.1268906722526...,2021,2021,2021-11-01,2021,,
2,17389,17390,3.030845,17389,17390,2020-04-21 19:26:06.158,2022-04-08 11:32:07.773,2020-07-01,21.0,17.398327,...,"LINESTRING (-51.04574 -4.22884, -51.04577 -4.2...",2,POLYGON ((-51.045497065730736 -4.2288636655155...,POLYGON ((-51.045520090020176 -4.2288490415850...,"2020,2022","2020,2022",2020-07-01,2020,,
3,6460,6461,3.554890,6460,6461,2020-08-01 03:18:01.761,2024-09-24 03:41:51.932,2023-11-01,10.0,415.099304,...,"LINESTRING (-49.80725 -5.00274, -49.80727 -5.0...",3,POLYGON ((-49.80700601158733 -5.00276303350481...,POLYGON ((-49.80702198733511 -5.00273565793518...,"2022,2023","2022,2023",2023-11-01,2023,2022,2022
4,7223,7224,3.939691,7223,7224,2020-09-21 07:02:58.924,2023-01-30 02:27:39.032,2022-09-01,4.0,55.437439,...,"LINESTRING (-49.64358 -4.82367, -49.64357 -4.8...",4,POLYGON ((-49.64333484050107 -4.82368899999371...,POLYGON ((-49.64332501718003 -4.82366065913984...,2022,2022,2022-09-01,2022,,


In [ ]:
# Unnecessary columns to drop
pairs = pairs_final.drop(columns=[
    "pre_buff_10", "post_buff_10",
    "fire_date_dt", "fire_date_year",
    "time_1_dt", "last_fire_year_for_time1", "source_file", "fire_years_before_fire_date"
], errors="ignore")

pairs.head()

,index_1,index_2,distance_m,pre_index,post_index,time_1,time_2,fire_date,post_fire_months,pre_agbd,...,delta_agbd,pre_geom,post_geom,pairing_crs,geometry,pair_uid,fire_frequency,fire_years,pre_veg_age,n_fire_years
0,36287,36288,2.041762,36287,36288,2020-01-26 05:35:53.896,2020-07-02 02:39:59.786,2020-04-01,3.0,147.778152,...,-186.904617,"POLYGON ((-49.643577 -4.614255, -49.64358 -4.6...","POLYGON ((-49.643561 -4.614246, -49.643563 -4....",EPSG:32722,"LINESTRING (-49.64374 -4.61424, -49.64372 -4.6...",0,1,2020,NaN,1
1,31053,31074,2.358854,31053,31074,2019-09-30 04:18:42.182,2022-08-07 23:45:11.162,2021-11-01,9.0,108.444229,...,24.885796,"POLYGON ((-50.255709 -3.126906, -50.255711 -3....","POLYGON ((-50.255704 -3.126885, -50.255706 -3....",EPSG:32722,"LINESTRING (-50.25587 -3.12689, -50.25586 -3.1...",1,1,2021,NaN,1
2,17389,17390,3.030845,17389,17390,2020-04-21 19:26:06.158,2022-04-08 11:32:07.773,2020-07-01,21.0,17.398327,...,15.451023,"POLYGON ((-51.045587 -4.228858, -51.045589 -4....","POLYGON ((-51.04561 -4.228843, -51.045612 -4.2...",EPSG:32722,"LINESTRING (-51.04574 -4.22884, -51.04577 -4.2...",2,2,"2020,2022",NaN,2
3,6460,6461,3.554890,6460,6461,2020-08-01 03:18:01.761,2024-09-24 03:41:51.932,2023-11-01,10.0,415.099304,...,414.003235,"POLYGON ((-49.807096 -5.002757, -49.807098 -5....","POLYGON ((-49.807112 -5.00273, -49.807114 -5.0...",EPSG:32722,"LINESTRING (-49.80725 -5.00274, -49.80727 -5.0...",3,2,"2022,2023",NaN,2
4,7223,7224,3.939691,7223,7224,2020-09-21 07:02:58.924,2023-01-30 02:27:39.032,2022-09-01,4.0,55.437439,...,3.437462,"POLYGON ((-49.643425 -4.823689, -49.643427 -4....","POLYGON ((-49.643415 -4.823655, -49.643417 -4....",EPSG:32722,"LINESTRING (-49.64358 -4.82367, -49.64357 -4.8...",4,1,2022,NaN,1


In [88]:
# Count the number of pairs in which the fire burned primary forest (# of NaN in pre_veg_age)
n_nan = pairs["pre_veg_age"].isna().sum()
print("Pairs with primary forest before:", int(n_nan))

Pairs with primary forest before: 8102


In [ ]:
# Checking in how many pairs fire_frequency is different from the number of years in fire_years

def count_years(s):
    if pd.isna(s):
        return 0
    s = str(s).strip()
    if s == "":
        return 0
    years = [t.strip() for t in s.split(",") if t.strip().isdigit()]
    return len(set(years))

pairs_final["n_fire_years"] = pairs_final["fire_years"].apply(count_years)

# Compare to fire_frequency
ff_num = pd.to_numeric(pairs_final["fire_frequency"], errors="coerce")

mismatch_mask = ff_num.ne(pairs_final["n_fire_years"])

print("Pairs where fire_frequency != number of years in fire_years:", int(mismatch_mask.sum()))
print("Total pairs:", len(pairs_final))
print("Percent:", round(100 * mismatch_mask.mean(), 2), "%")

# optional: inspect examples
pairs_final.loc[mismatch_mask, ["pair_uid", "fire_frequency", "fire_years", "n_fire_years"]].head(20)

Pairs where fire_frequency != number of years in fire_years: 6889
Total pairs: 22280
Percent: 30.92 %


,pair_uid,fire_frequency,fire_years,n_fire_years
5,5,3,"1999,2014,2018,2022",4
16,16,4,"2020,2021,2023",3
17,17,4,"2020,2021,2023",3
23,23,2,2022,1
24,24,2,2022,1
27,27,2,2022,1
28,28,2,2022,1
30,30,3,"2021,2022",2
32,32,2,2021,1
56,56,3,"2020,2023",2


In [ ]:
out_path_same = r"GEDI_Pairs/GEDI_Footprint_Pairs_Fire_FF_FYs.gpkg"
pairs.to_file(out_path_same, driver="GPKG")
print("Saved:", out_path_same)

Saved: GEDI_Pairs\GEDI_Footprint_Pairs_Fire_FF_FYs.gpkg
